# Post-hoc Proofreader Analysis from Tasks + DifferStack History

**Goal of this notebook:** demonstrate the *access patterns* for NeuVue
proofreading data and turn the raw `differ_stack` history into a first look at
the **"language of proofreading"** — the vocabulary, grammar, and per-person
dialects of proofreader behavior.

It shows how to:

1. Survey what data exists in the queue (namespaces, users, volume, date span).
2. Query task-level proofreading outcomes from NeuVue Queue.
3. Retrieve Neuroglancer `differ_stack` entries (the same change history used by "Rewind history").
4. Normalize both datasets into analysis-ready tables and **validate data quality**.
5. Characterize the **language of proofreading** and per-proofreader behavior.

Based on query patterns from:
- `aplbrain/neuvue-manage` `task-management/posthoc_proofreader_differstack_analysis.ipynb`
- `neuvueclient.NeuvueQueue` (`aplbrain/neuvue-client`)

### What is in here

| Section | Output |
|---|---|
| 3. Survey | namespaces / users / volume / date span across the DB |
| 4-8. Query & QC | task table + null audit + empirical `duration`-unit check |
| 9-12. DifferStack | normalized per-event table joined to task metadata |
| 13. Proofreader summary | workload, duration percentiles, event rates |
| **14. Language of proofreading** | **vocabulary (Zipf), grammar (transitions), dialects (divergence)** |
| 15-16. Viz & timing | distributions, action mix, throughput, inter-event rhythm |
| 17-19. Stats & QC | Kruskal-Wallis / Spearman, outliers, single-task timeline |

## Notes Before Running

- **Auth.** `NeuvueQueue` needs credentials: a `~/.neuvuequeue/neuvuequeue.cfg` (created by an interactive `login()` Auth0/Google flow) **or** `NEUVUEQUEUE_ACCESS_TOKEN` + `NEUVUEQUEUE_REFRESH_TOKEN` env vars. Run this where those exist (it cannot complete the browser login on a headless box).
- **No enumeration API.** There is no `get_namespaces()` / `get_users()`. Section 3 surveys scope by sampling recent active tasks with a `limit`.
- By default, `get_tasks(...)` converts Neuroglancer state URLs to JSON, which is slow. We disable this with `convert_states_to_json=False` for analytics.
- **`get_tasks()` injects `{"active": True}`** when `active` is unset, and **mutates datetime range bounds in place** (datetime to epoch ms). We pass `copy.deepcopy()` of the sieve so the original stays re-runnable.
- **`duration` is stored in milliseconds** (like `created` / `opened` / `closed`). We convert to seconds and **empirically validate** the unit against `closed - opened`.
- The DB stores `differ_stack` as a free-form list of maps (`[Map]`); the event shape is set by the Neuroglancer "differ", not the server. We flatten defensively and **discover the real keys at runtime** (Section 11). It is normal for some tasks to have no differstack; those still appear with `n_events = 0`.

## 0. Imports & Setup

In [ ]:
from __future__ import annotations

import os
import sys
import re
import copy
import json
import urllib.parse
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd

# Plotting (optional but recommended).
try:
    import matplotlib.pyplot as plt
    HAVE_MPL = True
    plt.rcParams["figure.figsize"] = (9, 4.5)
    plt.rcParams["axes.grid"] = True
    plt.rcParams["grid.alpha"] = 0.3
except Exception as exc:
    HAVE_MPL = False
    print(f"matplotlib unavailable ({exc}); plots will be skipped.")

# Optional: non-parametric statistical tests.
try:
    import scipy.stats as sstats
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)
print(f"pandas {pd.__version__} | matplotlib={HAVE_MPL} | scipy={HAVE_SCIPY}")

## 1. Locate the NeuVue Client

Walk upward from the working directory to find the repo root (and the vendored
`neuvue-client` checkout, if present). Falls back gracefully when `neuvueclient`
is installed as a normal package.

In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    """Return the repo root by walking upward from `start`.

    Prefers a directory that contains the vendored neuvue-client checkout, then
    a `.git` directory, then the current working directory.
    """
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for base in candidates:
        if (base / "neuvue_project" / "deps" / "neuvue-client").exists():
            return base
    for base in candidates:
        if (base / ".git").exists():
            return base
    return start


REPO_ROOT = find_repo_root()
CLIENT_PATH = REPO_ROOT / "neuvue_project" / "deps" / "neuvue-client"
if CLIENT_PATH.exists() and str(CLIENT_PATH) not in sys.path:
    sys.path.insert(0, str(CLIENT_PATH))

from neuvueclient import NeuvueQueue

print(f"REPO_ROOT:   {REPO_ROOT}")
print(f"CLIENT_PATH: {CLIENT_PATH} (exists={CLIENT_PATH.exists()})")

## 2. Configure Query Scope

Set these values for your analysis window. Tips:
- Keep the first query narrow (single namespace + recent date range) to iterate quickly.
- Not sure what to put here? Run **Section 3** first to see what namespaces / users exist.

In [ ]:
QUEUE_URL = os.getenv("NEUVUE_QUEUE_URL", "https://queue.neuvue.io")

NAMESPACE = "forcedChoiceExample"        # change me (see Section 3)
ASSIGNEES = ["dxenes1"]                  # set [] for all assignees in namespace
TASK_STATUSES = ["closed", "errored"]    # typical submitted outcomes in this app

# Optional closed-time filter. Use None for an open-ended range.
CLOSED_AFTER = dt.datetime(2024, 1, 1)
CLOSED_BEFORE = None

# NeuVue stores task `duration` in milliseconds (same as created/opened/closed).
# Section 7 validates this empirically; set to "s" if your deployment differs.
DURATION_UNIT = "ms"   # "ms" or "s"

# Treat differ-event gaps longer than this as idle/away time (Section 16).
IDLE_GAP_S = 120.0

# Keep the payload light for analytics. NOTE: created/opened/closed conversions
# inside get_tasks() are guarded by column presence, so trimming this is safe.
TASK_SELECT = [
    "assignee", "author", "namespace", "status", "priority", "duration",
    "created", "opened", "closed", "metadata", "seg_id", "tags", "ng_state",
]

In [ ]:
client = NeuvueQueue(QUEUE_URL)
print(f"Connected to: {QUEUE_URL}")

## 3. Survey Available Data (Access Pattern)

There is **no enumeration endpoint** (`get_namespaces`/`get_users` do not exist),
so we discover scope by sampling the most recent active tasks with a `limit` and
aggregating client-side. This is the access pattern to answer "how much data, for
how many users, over what time span?" — raise `SURVEY_LIMIT` for fuller coverage.

In [ ]:
SURVEY_LIMIT = 20000   # bounded sample; depaginate stops once this many rows arrive

survey = client.get_tasks(
    sieve={"active": True},
    select=["namespace", "assignee", "status", "created", "closed", "duration"],
    convert_states_to_json=False,
    sort="-created",
    limit=SURVEY_LIMIT,
    pageSize=1000,
)
print(f"Sampled {len(survey):,} recent active tasks (limit={SURVEY_LIMIT}).")

if survey.empty:
    print("No tasks returned. Check credentials / queue URL.")
else:
    created = pd.to_datetime(survey["created"], errors="coerce")
    print(f"Distinct namespaces: {survey['namespace'].nunique():,} | "
          f"distinct assignees: {survey['assignee'].nunique():,}")
    print(f"Created span: {created.min()} -> {created.max()}")

    scope = (
        survey.groupby("namespace")
        .agg(n_tasks=("assignee", "size"), n_users=("assignee", "nunique"))
        .sort_values("n_tasks", ascending=False)
    )
    print()
    print("Top namespaces (tasks x distinct users):")
    display(scope.head(25))
    print("Most active assignees in the sample:")
    display(survey["assignee"].value_counts().rename("n_tasks").head(25))

## 4. Build Task Sieve

This follows the same `sieve` style used in the example query notebook. Multiple
assignees / statuses use an explicit `$in` (the same Mongo-style operator the
differstack query relies on).

In [ ]:
def build_task_sieve(
    namespace: str,
    assignees: list[str] | None = None,
    statuses: list[str] | None = None,
    closed_after: dt.datetime | None = None,
    closed_before: dt.datetime | None = None,
) -> dict:
    """Construct a Mongo-style sieve for NeuvueQueue.get_tasks().

    Note: get_tasks() injects {"active": True} when `active` is unset and mutates
    datetime bounds in place (datetime to epoch ms). Always pass a fresh copy to
    get_tasks() so the original dict stays re-runnable.
    """
    sieve: dict = {"namespace": namespace}

    if assignees:
        sieve["assignee"] = assignees[0] if len(assignees) == 1 else {"$in": list(assignees)}

    if statuses:
        sieve["status"] = statuses[0] if len(statuses) == 1 else {"$in": list(statuses)}

    if closed_after is not None or closed_before is not None:
        closed_query: dict = {}
        if closed_after is not None:
            closed_query["$gt"] = closed_after
        if closed_before is not None:
            closed_query["$lt"] = closed_before
        sieve["closed"] = closed_query

    return sieve


task_sieve = build_task_sieve(
    namespace=NAMESPACE,
    assignees=ASSIGNEES,
    statuses=TASK_STATUSES,
    closed_after=CLOSED_AFTER,
    closed_before=CLOSED_BEFORE,
)

task_sieve

## 5. Query Tasks

`convert_states_to_json=False` avoids expensive state download and is usually
better for post-hoc behavior analysis. We deep-copy the sieve because
`get_tasks()` rewrites datetime bounds in place.

In [ ]:
tasks = client.get_tasks(
    sieve=copy.deepcopy(task_sieve),   # deepcopy: get_tasks mutates datetime bounds
    select=TASK_SELECT,
    convert_states_to_json=False,
    pageSize=1000,
)

print(f"Fetched tasks: {len(tasks):,}")
tasks.head(3)

## 6. Post-process Tasks

In [ ]:
if tasks.empty:
    raise RuntimeError("No tasks matched your sieve. Adjust namespace/assignee/date filters.")

# Move the Mongo _id index to an explicit column for joins with differstacks.
tasks = tasks.reset_index(names="task_id")
tasks["task_id"] = tasks["task_id"].astype(str)

# Duration -> seconds / minutes (unit validated in Section 7).
_dur_div = 1000.0 if DURATION_UNIT == "ms" else 1.0
tasks["duration_s"] = pd.to_numeric(tasks["duration"], errors="coerce") / _dur_div
tasks["duration_min"] = tasks["duration_s"] / 60.0

# Pull common forced-choice metadata fields out of the nested dict.
def _meta_get(meta, key):
    return meta.get(key) if isinstance(meta, dict) else None

if "metadata" in tasks.columns:
    for field in ["decision", "flag_reason"]:
        tasks[field] = tasks["metadata"].apply(lambda m: _meta_get(m, field))

# Convenience date column for temporal grouping.
if "closed" in tasks.columns:
    tasks["closed_date"] = pd.to_datetime(tasks["closed"], errors="coerce").dt.date

preview = [c for c in ["task_id", "assignee", "status", "duration_s", "decision", "closed"] if c in tasks.columns]
tasks[preview].head(10)

## 7. Data Quality & Duration-Unit Validation

Before trusting any summary, audit nulls and confirm the `duration` unit by
comparing it against the wall-clock span `closed - opened`.

In [ ]:
# 1) Null / non-null overview for the columns we rely on.
key_cols = [c for c in ["assignee", "status", "duration", "created", "opened", "closed",
                        "decision", "flag_reason", "seg_id"] if c in tasks.columns]
quality = pd.DataFrame({
    "n_nonnull": tasks[key_cols].notna().sum(),
    "n_null": tasks[key_cols].isna().sum(),
    "pct_null": (tasks[key_cols].isna().mean() * 100).round(1),
})
display(quality)

# 2) Empirically validate the duration unit against (closed - opened).
if {"closed", "opened"}.issubset(tasks.columns):
    span_s = (pd.to_datetime(tasks["closed"]) - pd.to_datetime(tasks["opened"])).dt.total_seconds()
    ok = span_s.notna() & (span_s > 0) & tasks["duration"].notna() & (tasks["duration"] > 0)
    if ok.any():
        ratio = float((tasks.loc[ok, "duration"] / span_s[ok]).median())
        inferred = "ms" if ratio > 100 else ("s" if ratio < 10 else "ambiguous")
        print(f"median(duration / (closed-opened)_seconds) = {ratio:,.1f}")
        print(f"  -> raw `duration` looks like it is in: {inferred}")
        print(f"  -> configured DURATION_UNIT = {DURATION_UNIT!r}")
        if inferred not in ("ambiguous", DURATION_UNIT):
            print(f"  WARNING: configured unit {DURATION_UNIT!r} disagrees with inferred "
                  f"{inferred!r}. Update DURATION_UNIT and re-run Section 6.")
    else:
        print("Not enough rows with positive duration and span to validate the unit.")

# 3) Suspicious durations.
n_zero = int((tasks["duration_s"] <= 0).sum())
n_huge = int((tasks["duration_s"] > 3600).sum())   # open longer than 1 hour
print()
print(f"Tasks with duration_s <= 0: {n_zero}")
print(f"Tasks with duration_s > 1h: {n_huge}")

## 8. Quick Task-Level Sanity Checks

In [ ]:
display(tasks["status"].value_counts(dropna=False).rename("n_tasks"))

if "decision" in tasks.columns and tasks["decision"].notna().any():
    display(tasks["decision"].value_counts(dropna=False).rename("n_tasks"))

display(
    tasks.groupby("assignee", dropna=False)
    .agg(
        n_tasks=("task_id", "nunique"),
        median_duration_s=("duration_s", "median"),
        mean_duration_s=("duration_s", "mean"),
        total_hours=("duration_s", lambda s: s.sum() / 3600.0),
    )
    .sort_values("n_tasks", ascending=False)
    .round(1)
)

## 9. Pull DifferStack History for These Tasks

The client lists differ stacks via `get_differ_stacks(...)`. We query by task IDs
in **small** chunks (~20): differstack payloads are large, so a big `task_id`
`$in` query overruns the gateway and returns **502** (25+ tends to fail). A
per-task fallback covers deployments that reject `$in` entirely. Coverage is
typically partial — many tasks have no differstack.

In [ ]:
def chunks(items: list[str], chunk_size: int):
    for i in range(0, len(items), chunk_size):
        yield items[i : i + chunk_size]


def get_differstacks_for_tasks(client: NeuvueQueue, task_ids: list[str], chunk_size: int = 20) -> pd.DataFrame:
    # NOTE: keep chunk_size small (~20). Differstack payloads are large, so a big
    # task_id $in query overruns the gateway and returns 502. 25+ tends to fail.
    frames = []
    task_ids = [str(t) for t in task_ids]

    for ci, task_chunk in enumerate(chunks(task_ids, chunk_size)):
        sieve = {"task_id": {"$in": task_chunk}, "active": True}
        try:
            ds = client.get_differ_stacks(sieve=sieve, pageSize=1000)
            if len(ds) > 0:
                frames.append(ds)
        except Exception as chunk_exc:
            # Fallback: per-task queries when the deployment does not support $in here.
            print(f"Chunk {ci} query failed ({chunk_exc}). Falling back to per-task queries.")
            for tid in task_chunk:
                try:
                    ds_single = client.get_differ_stacks(sieve={"task_id": tid, "active": True}, pageSize=1000)
                    if len(ds_single) > 0:
                        frames.append(ds_single)
                except Exception as item_exc:
                    print(f"  task_id={tid}: {item_exc}")

    if not frames:
        return pd.DataFrame(columns=["task_id", "differ_stack"])  # consistent shape

    out = pd.concat(frames, axis=0)
    out = out[~out.index.duplicated(keep="first")]
    return out


differstacks = get_differstacks_for_tasks(client, tasks["task_id"].tolist(), chunk_size=20)
n_tasks_with_ds = differstacks["task_id"].astype(str).nunique() if not differstacks.empty else 0
print(f"Fetched differstack rows: {len(differstacks):,} "
      f"covering {n_tasks_with_ds:,} / {tasks['task_id'].nunique():,} tasks")
differstacks.head(3)

## 10. Normalize DifferStack Entries to an Event Table

Each `differ_stack` row is a list of change events from a proofreading session.
On `queue.neuvue.io` the real event is `{patch, timestamp, change}`: `change` is
the constant `"Change"`, `timestamp` is epoch-ms, and **`patch` is a
diff-match-patch over the URL-encoded Neuroglancer state** — the literal edit.
`classify_patch()` turns each patch into a first-pass action label
(navigation / annotation / segment_edit / other). We still flatten defensively
since the payload is free-form (`[Map]`) and varies by Neuroglancer build.

In [ ]:
def _coerce_event_to_dict(event):
    if isinstance(event, dict):
        return event
    if isinstance(event, list):
        return {"raw_event": event}
    if event is None:
        return {}
    return {"raw_event": event}


def _pick_first_existing(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None


def _parse_event_time(series: pd.Series) -> pd.Series:
    """Parse epoch s/ms/us/ns or string timestamps into tz-aware datetimes."""
    numeric = pd.to_numeric(series, errors="coerce")
    if numeric.notna().mean() > 0.6:
        med = numeric.dropna().median() if numeric.notna().any() else np.nan
        if pd.isna(med):
            unit = "s"
        elif med >= 1e17:
            unit = "ns"
        elif med >= 1e14:
            unit = "us"
        elif med >= 1e11:
            unit = "ms"
        else:
            unit = "s"
        return pd.to_datetime(numeric, unit=unit, errors="coerce", utc=True)
    return pd.to_datetime(series, errors="coerce", utc=True)


def classify_patch(patch) -> str:
    """First-pass action label for a real NeuVue differ event.

    Live events are {patch, timestamp, change}; `change` is the constant "Change"
    and `patch` is a diff-match-patch over the URL-encoded Neuroglancer state. The
    changed field name is often outside the small diff window, so numeric-only
    diffs (coordinate / camera changes) are treated as navigation. Heuristic --
    refine once you inspect real payloads in Section 11.
    """
    d = urllib.parse.unquote(str(patch))
    if any(k in d for k in ("annotation", "description", "tagId", '"tags"', "spine")):
        return "annotation"
    if any(k in d for k in ("segments", "equivalences", "hiddenSegments")):
        return "segment_edit"
    if any(k in d for k in ('"position"', "Orientation", "Scale", "zoomFactor", "Zoom", "voxelCoordinates")):
        return "navigation"
    body = re.sub(r"@@[^@]*@@", "", d)
    if len(re.sub(r"[^A-Za-z]", "", body)) <= 4:   # numeric-only diff -> navigation
        return "navigation"
    return "other"


def normalize_differstack_events(differ_df: pd.DataFrame) -> pd.DataFrame:
    base_cols = ["differ_id", "task_id", "event_order", "action", "event_time"]
    if differ_df.empty:
        return pd.DataFrame(columns=base_cols)

    temp = differ_df.reset_index(names="differ_id").copy()

    # Ensure list-like values; empty/None -> empty list so explode is predictable.
    temp["differ_stack"] = temp["differ_stack"].apply(
        lambda x: x if isinstance(x, list) else ([] if (x is None or (np.isscalar(x) and pd.isna(x))) else [x])
    )

    exploded = temp.explode("differ_stack", ignore_index=True)
    exploded = exploded[exploded["differ_stack"].notna()].copy()
    if exploded.empty:
        return pd.DataFrame(columns=base_cols)
    exploded["event_order"] = exploded.groupby("differ_id").cumcount()

    payload = exploded["differ_stack"].apply(_coerce_event_to_dict)
    flat = pd.json_normalize(payload.tolist())
    # Avoid duplicate join-key columns if a payload happens to carry them.
    flat = flat.loc[:, [c for c in flat.columns if c not in {"differ_id", "task_id", "event_order"}]]

    meta = exploded[["differ_id", "task_id", "event_order"]].reset_index(drop=True)
    events = pd.concat([meta, flat.reset_index(drop=True)], axis=1)

    action_candidates = ["action", "type", "event", "operation", "op", "kind", "name",
                         "changeType", "change_type"]
    time_candidates = ["timestamp", "time", "ts", "created", "created_at", "updated",
                       "updated_at", "date"]

    action_col = _pick_first_existing(events.columns, action_candidates)
    time_col = _pick_first_existing(events.columns, time_candidates)

    if "patch" in events.columns:                 # real NeuVue schema: diff-match-patch
        events["action"] = events["patch"].apply(classify_patch)
    elif action_col is not None:
        events["action"] = events[action_col].astype(str)
    else:
        events["action"] = "unknown"
    events["event_time"] = _parse_event_time(events[time_col]) if time_col is not None else pd.NaT
    events["task_id"] = events["task_id"].astype(str)
    return events


events = normalize_differstack_events(differstacks)
print(f"Normalized events: {len(events):,} | columns: {len(events.columns)}")
display(events.head(5))

## 11. Inspect Event Schema

The differstack payload is free-form, so confirm which keys actually exist and
which one we picked as `action` / `event_time`. Tune the candidate lists in
`normalize_differstack_events()` if your build stores them under other keys.

In [ ]:
event_meta = {"differ_id", "task_id", "event_order", "action", "event_time"}
payload_cols = [c for c in events.columns if c not in event_meta]
print(f"Discovered {len(payload_cols)} payload field(s) in differ events:")
display(pd.Series(payload_cols, name="event_payload_columns"))

if not events.empty:
    print()
    print("Inferred action distribution:")
    display(events["action"].value_counts(dropna=False).head(20).rename("n_events"))
    print(f"Parsed event_time non-null: {events['event_time'].notna().mean():.1%}")
    if payload_cols:
        nonempty = events.loc[events[payload_cols].notna().any(axis=1), payload_cols]
        if not nonempty.empty:
            print()
            print("Sample event payload (first non-empty):")
            print(json.dumps(nonempty.iloc[0].dropna().to_dict(), indent=2, default=str))

## 12. Join Events Back to Task Metadata

In [ ]:
task_cols_for_join = [
    c for c in [
        "task_id", "assignee", "author", "namespace", "status", "priority",
        "duration", "duration_s", "duration_min", "created", "opened", "closed",
        "closed_date", "decision", "flag_reason", "seg_id", "tags",
    ]
    if c in tasks.columns
]

events_enriched = events.merge(
    tasks[task_cols_for_join],
    on="task_id",
    how="left",
    validate="m:1",
)
display(events_enriched.head(5))

## 13. Per-proofreader Summary

A practical starting point for behavior analysis: workload, duration percentiles,
fraction of tasks with differstack data, and event rates per proofreader.

In [ ]:
if not events_enriched.empty:
    events_per_task = (
        events_enriched.groupby("task_id", dropna=False).size().rename("n_events").reset_index()
    )
else:
    events_per_task = pd.DataFrame({"task_id": pd.Series(dtype=str), "n_events": pd.Series(dtype=int)})

tasks_aug = tasks.merge(events_per_task, on="task_id", how="left")
tasks_aug["n_events"] = tasks_aug["n_events"].fillna(0).astype(int)
tasks_aug["has_differ"] = tasks_aug["n_events"] > 0


def _p25(s): return s.quantile(0.25)
def _p75(s): return s.quantile(0.75)
def _p90(s): return s.quantile(0.90)


proofreader_summary = (
    tasks_aug.groupby("assignee", dropna=False)
    .agg(
        n_tasks=("task_id", "nunique"),
        pct_with_differ=("has_differ", lambda s: round(100 * s.mean(), 1)),
        total_hours=("duration_s", lambda s: round(s.sum() / 3600.0, 2)),
        dur_p25_s=("duration_s", _p25),
        dur_med_s=("duration_s", "median"),
        dur_p75_s=("duration_s", _p75),
        dur_p90_s=("duration_s", _p90),
        mean_events=("n_events", "mean"),
        median_events=("n_events", "median"),
    )
    .sort_values("n_tasks", ascending=False)
    .round(1)
)
display(proofreader_summary)

## 14. The Language of Proofreading

Treat each session's `differ_stack` as an *utterance*: a time-ordered sequence of
edit events. Three views:

- **Vocabulary** — which action types occur, and how often (a Zipf-like view).
- **Grammar** — how actions follow one another within a session (transition structure).
- **Dialects** — how that vocabulary/grammar differs *between* proofreaders.

This is the data-anchor for "calibrate the people": competence and agreement leave
signatures in this language, and *early* language is what a predictive test reads.

> If Section 11 shows the real edits live under different keys (e.g. added vs.
> removed segment sets = merges vs. splits), adapt `classify_action()` below — the
> rest of this section keys off whatever `action` column you produce.

**Observed on `queue.neuvue.io` (2026-06, `fullyProofread`):** 112 sessions /
7,203 events / 13 proofreaders; ~32 events per session (median, up to 605);
first-pass vocabulary ~70% navigation, ~10% annotation, <1% explicit segment
edits; and clear per-person dialects (navigation share 28–83%, annotation 4–33%,
label-entropy 0.8–1.6 bits). The DB-wide sample reached ~78 proofreaders across
19 namespaces. Sharper merge/split detection needs state reconstruction (above).

In [ ]:
# `action` is already a first-pass label from classify_patch() (navigation /
# annotation / segment_edit / other). To sharpen the vocabulary -- especially to
# split merges from splits -- reconstruct before/after state by applying the
# diff-match-patch and diff the `segments` sets, then override here.
def classify_action(row) -> str:
    return str(row.get("action", "unknown"))


if not events_enriched.empty:
    events_enriched["action_class"] = events_enriched.apply(classify_action, axis=1)
    LANG_COL = "action_class"
else:
    LANG_COL = "action"
print(f"Language token column: {LANG_COL!r}")

**14a. Vocabulary** — the tokens of proofreading and their frequency.

In [ ]:
if events_enriched.empty:
    print("No differ events: the language of proofreading is empty for this sieve.")
else:
    vocab = events_enriched[LANG_COL].value_counts()
    print(f"Vocabulary size (distinct actions): {vocab.shape[0]:,}")
    print(f"Total events (tokens): {int(vocab.sum()):,}")
    display(vocab.rename("n_events").head(25))
    if HAVE_MPL and vocab.shape[0] > 1:
        ranks = np.arange(1, vocab.shape[0] + 1)
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.loglog(ranks, vocab.values, marker="o", ls="none", color="darkred")
        ax.set(title="Action rank-frequency (Zipf view)", xlabel="rank", ylabel="frequency")
        fig.tight_layout()
        plt.show()

**14b. Grammar** — how one action follows another within a session (bigram transitions).

In [ ]:
if events_enriched.empty or events_enriched[LANG_COL].nunique() < 2:
    print("Need >=2 action types within ordered sessions to model grammar.")
else:
    seq = events_enriched.sort_values(["differ_id", "event_order"]).copy()
    seq["next_action"] = seq.groupby("differ_id")[LANG_COL].shift(-1)
    bigrams = seq.dropna(subset=["next_action"])
    topk = events_enriched[LANG_COL].value_counts().head(10).index
    bg = bigrams[bigrams[LANG_COL].isin(topk) & bigrams["next_action"].isin(topk)]
    if bg.empty:
        print("No within-session consecutive action pairs among the top actions.")
    else:
        trans = (
            bg.groupby([LANG_COL, "next_action"]).size().unstack(fill_value=0)
            .reindex(index=topk, columns=topk, fill_value=0)
        )
        trans_norm = trans.div(trans.sum(axis=1).replace(0, np.nan), axis=0)
        print("P(next action | current action), top actions:")
        display(trans_norm.round(2))
        print("Most common consecutive pairs:")
        display(bg.groupby([LANG_COL, "next_action"]).size().sort_values(ascending=False).head(12).rename("n"))
        if HAVE_MPL:
            fig, ax = plt.subplots(figsize=(1.4 + 0.7 * len(topk), 1.2 + 0.6 * len(topk)))
            im = ax.imshow(trans_norm.fillna(0).values, cmap="magma", aspect="auto", vmin=0, vmax=1)
            ax.set_xticks(range(len(topk)))
            ax.set_xticklabels(trans_norm.columns, rotation=45, ha="right")
            ax.set_yticks(range(len(topk)))
            ax.set_yticklabels(trans_norm.index)
            ax.set(title="Grammar: action -> next action", xlabel="next", ylabel="current")
            fig.colorbar(im, ax=ax, label="P(next | current)")
            fig.tight_layout()
            plt.show()

**14c. Dialects** — per-proofreader vocabulary, diversity (entropy), and pairwise
distance (Jensen-Shannon divergence of action mix). High divergence = different
"dialects"; this is the measurable raw material for calibrating annotators.

In [ ]:
if events_enriched.empty:
    print("No events for dialect analysis.")
else:
    counts = events_enriched.groupby(["assignee", LANG_COL]).size().unstack(fill_value=0)
    probs = counts.div(counts.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0)

    def _shannon_bits(p):
        p = p[p > 0]
        return float(-(p * np.log2(p)).sum())

    dialect = pd.DataFrame({
        "n_events": counts.sum(axis=1).astype(int),
        "vocab_size": (counts > 0).sum(axis=1).astype(int),
        "entropy_bits": probs.apply(_shannon_bits, axis=1),
    }).sort_values("n_events", ascending=False)
    print("Per-proofreader dialect summary:")
    display(dialect.head(25).round(2))

    def _js_div(p, q):
        m = 0.5 * (p + q)
        def _kl(a, b):
            mask = a > 0
            return float((a[mask] * np.log2(a[mask] / b[mask])).sum())
        return 0.5 * _kl(p, m) + 0.5 * _kl(q, m)

    top_users = dialect.head(8).index.tolist()
    if len(top_users) >= 2:
        D = pd.DataFrame(index=top_users, columns=top_users, dtype=float)
        for a in top_users:
            for b in top_users:
                D.loc[a, b] = _js_div(probs.loc[a].values, probs.loc[b].values)
        print("Dialect distance (Jensen-Shannon divergence, bits) among most active users:")
        display(D.round(3))
        if HAVE_MPL:
            fig, ax = plt.subplots(figsize=(1.4 + 0.7 * len(top_users), 1.2 + 0.6 * len(top_users)))
            im = ax.imshow(D.values.astype(float), cmap="viridis")
            ax.set_xticks(range(len(top_users)))
            ax.set_xticklabels(top_users, rotation=45, ha="right")
            ax.set_yticks(range(len(top_users)))
            ax.set_yticklabels(top_users)
            ax.set(title="Dialect distance (JS divergence of action mix)")
            fig.colorbar(im, ax=ax, label="JS divergence (bits)")
            fig.tight_layout()
            plt.show()
    else:
        print("Need >=2 proofreaders with events for a dialect-distance map.")

## 15. Visualizations

Distributions and per-proofreader comparisons. Each cell degrades gracefully when
matplotlib is missing or there is no differstack data.

In [ ]:
# Task duration distribution (linear + log10 to handle the heavy right tail).
if HAVE_MPL:
    d = tasks_aug["duration_s"].dropna()
    d = d[d > 0]
    if not d.empty:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].hist(d, bins=40, color="steelblue", edgecolor="white")
        axes[0].axvline(d.median(), color="crimson", ls="--", label=f"median={d.median():.0f}s")
        axes[0].set(title="Task duration", xlabel="seconds", ylabel="n_tasks")
        axes[0].legend()
        axes[1].hist(np.log10(d), bins=40, color="seagreen", edgecolor="white")
        axes[1].set(title="Task duration (log10)", xlabel="log10(seconds)", ylabel="n_tasks")
        fig.tight_layout()
        plt.show()
        display(d.describe(percentiles=[.1, .25, .5, .75, .9, .99]).round(1).rename("duration_s"))
    else:
        print("No positive durations to plot.")
else:
    print("matplotlib not available.")

In [ ]:
# Differ events per task, and events-vs-duration relationship.
if HAVE_MPL:
    ev = tasks_aug["n_events"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    nb = int(ev.max()) if len(ev) else 0
    bins = range(0, nb + 2) if 0 < nb <= 40 else 40
    axes[0].hist(ev, bins=bins, color="indigo", edgecolor="white")
    axes[0].set(title="Differ events per task", xlabel="n_events", ylabel="n_tasks")

    sub = tasks_aug[tasks_aug["duration_s"] > 0]
    axes[1].scatter(sub["duration_s"], sub["n_events"], alpha=0.4, s=18, color="darkorange")
    title = "Events vs duration"
    if len(sub) > 2 and sub["n_events"].nunique() > 1:
        r = sub[["duration_s", "n_events"]].corr(method="spearman").iloc[0, 1]
        title = f"Events vs duration (Spearman r={r:.2f})"
    axes[1].set(title=title, xlabel="duration_s", ylabel="n_events")

    fig.tight_layout()
    plt.show()
else:
    print("matplotlib not available.")

In [ ]:
# Per-proofreader comparison bars.
if HAVE_MPL and not proofreader_summary.empty:
    ps = proofreader_summary
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    ps["n_tasks"].plot(kind="bar", ax=axes[0], color="steelblue", title="Tasks per proofreader")
    ps["dur_med_s"].plot(kind="bar", ax=axes[1], color="darkorange", title="Median duration (s)")
    ps["mean_events"].plot(kind="bar", ax=axes[2], color="seagreen", title="Mean events / task")
    for ax in axes:
        ax.set_xlabel("assignee")
        ax.tick_params(axis="x", rotation=45)
    fig.tight_layout()
    plt.show()
else:
    print("Nothing to plot (no matplotlib or empty summary).")

In [ ]:
# Duration distribution by proofreader (top 12 by task count, outliers hidden).
if HAVE_MPL:
    top = tasks_aug["assignee"].value_counts().head(12).index.tolist()
    groups = []
    for a in top:
        g = tasks_aug.loc[(tasks_aug["assignee"] == a) & (tasks_aug["duration_s"] > 0), "duration_s"].values
        if len(g) > 0:
            groups.append((str(a), g))
    if groups:
        fig, ax = plt.subplots(figsize=(min(2 + len(groups), 14), 4.5))
        ax.boxplot([g for _, g in groups], showfliers=False)
        ax.set_xticks(range(1, len(groups) + 1))
        ax.set_xticklabels([a for a, _ in groups], rotation=45, ha="right")
        ax.set(title="Duration distribution by proofreader (outliers hidden)", ylabel="duration_s")
        fig.tight_layout()
        plt.show()
    else:
        print("No positive durations to plot.")
else:
    print("matplotlib not available.")

In [ ]:
# Throughput over time (tasks closed per day) and cumulative total.
if "closed_date" in tasks_aug.columns and tasks_aug["closed_date"].notna().any():
    per_day = tasks_aug.dropna(subset=["closed_date"]).groupby("closed_date").size().rename("n_tasks")
    per_day.index = pd.to_datetime(per_day.index)
    per_day = per_day.sort_index()
    display(per_day.describe().round(1).rename("tasks_per_day"))
    if HAVE_MPL and not per_day.empty:
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        axes[0].bar(per_day.index, per_day.values, color="slateblue")
        axes[0].set(title="Tasks closed per day", ylabel="n_tasks")
        axes[1].plot(per_day.index, per_day.cumsum().values, color="darkgreen", marker="o", ms=3)
        axes[1].set(title="Cumulative tasks closed", ylabel="cumulative")
        for ax in axes:
            ax.tick_params(axis="x", rotation=45)
        fig.autofmt_xdate()
        fig.tight_layout()
        plt.show()
else:
    print("No `closed_date` available for temporal analysis.")

## 16. Session Timing — Fluency

Only meaningful when differ events carry timestamps. We measure inter-event gaps
(the *rhythm* of proofreading), the session span (last - first event), and
"active" time (sum of gaps under the `IDLE_GAP_S` idle threshold).

In [ ]:
if not events_enriched.empty and events_enriched["event_time"].notna().any():
    ev = events_enriched.dropna(subset=["event_time"]).copy()
    ev = ev.sort_values(["differ_id", "event_time"])
    ev["gap_s"] = ev.groupby("differ_id")["event_time"].diff().dt.total_seconds()

    sess = (
        ev.groupby(["differ_id", "task_id", "assignee"], dropna=False)
        .agg(
            n_events=("event_order", "size"),
            span_s=("event_time", lambda s: (s.max() - s.min()).total_seconds()),
        )
        .reset_index()
    )
    active = (
        ev[ev["gap_s"].notna() & (ev["gap_s"] <= IDLE_GAP_S)]
        .groupby("differ_id")["gap_s"].sum().rename("active_s")
    )
    sess = sess.merge(active, on="differ_id", how="left")
    sess["active_s"] = sess["active_s"].fillna(0.0)
    sess = sess.merge(tasks_aug[["task_id", "duration_s"]], on="task_id", how="left")

    print("Per-session timing (one row per differ stack):")
    display(sess[["assignee", "n_events", "span_s", "active_s", "duration_s"]].describe().round(1))

    if HAVE_MPL:
        gaps = ev["gap_s"].dropna()
        gaps = gaps[gaps >= 0]
        if not gaps.empty:
            hi = gaps.quantile(0.99)
            gaps = gaps[gaps <= hi] if hi > 0 else gaps
            fig, ax = plt.subplots(figsize=(9, 4))
            ax.hist(gaps, bins=50, color="chocolate", edgecolor="white")
            ax.axvline(IDLE_GAP_S, color="crimson", ls="--", label=f"idle threshold={IDLE_GAP_S:.0f}s")
            ax.set(title="Inter-event gaps within sessions (<= p99)",
                   xlabel="seconds between events", ylabel="count")
            ax.legend()
            fig.tight_layout()
            plt.show()
else:
    print("Differ events have no parsed timestamps; skipping session-timing analysis.")
    print("Tune the time-field detection in normalize_differstack_events() if your")
    print("payload stores time under a different key.")

## 17. Optional Statistical Tests

Non-parametric tests (require `scipy`). Useful sanity checks before reading too
much into raw averages.

In [ ]:
if HAVE_SCIPY:
    # Do proofreaders differ in task duration? (Kruskal-Wallis; non-parametric)
    groups = [
        g["duration_s"].dropna().values
        for _, g in tasks_aug.groupby("assignee")
        if g["duration_s"].notna().sum() >= 5
    ]
    if len(groups) >= 2:
        H, p = sstats.kruskal(*groups)
        print(f"Kruskal-Wallis (duration_s across proofreaders): H={H:.2f}, p={p:.3g}")
        print("  -> proofreaders differ significantly in duration."
              if p < 0.05 else "  -> no significant difference detected.")
    else:
        print("Need >=2 proofreaders with >=5 tasks each for Kruskal-Wallis.")

    # Does interaction volume scale with time on task?
    sub = tasks_aug[tasks_aug["duration_s"] > 0]
    if sub["n_events"].nunique() > 1 and len(sub) >= 5:
        rho, p = sstats.spearmanr(sub["duration_s"], sub["n_events"])
        print()
        print(f"Spearman(duration_s, n_events): rho={rho:.2f}, p={p:.3g}")
    else:
        print("Not enough variation in n_events for a correlation test.")
else:
    print("scipy not installed; skipping statistical tests (pip install scipy).")

## 18. Outlier / QC Review

Surface tasks worth a manual look: extreme durations, extreme event counts, and
tasks missing differstack data.

In [ ]:
qc = tasks_aug.copy()
dur = qc["duration_s"]
dur_hi = dur.quantile(0.99)

if (qc["n_events"] > 0).any():
    evt_hi = qc["n_events"].quantile(0.99)
    high_evt = qc["n_events"] > evt_hi
else:
    high_evt = pd.Series(False, index=qc.index)

flags = pd.DataFrame({
    "very_long_duration": dur > dur_hi,
    "zero_or_neg_duration": dur <= 0,
    "high_event_count": high_evt,
    "no_differ_data": ~qc["has_differ"],
})
display(flags.sum().rename("n_tasks"))

outlier_cols = [c for c in ["task_id", "assignee", "status", "duration_s", "n_events",
                            "decision", "flag_reason"] if c in qc.columns]
print("Longest tasks:")
display(qc.nlargest(10, "duration_s")[outlier_cols])
print("Most differ events:")
display(qc.nlargest(10, "n_events")[outlier_cols])

## 19. Single-task Timeline

Deep-dive into one session. We pick the task with the most differ events so the
timeline is non-trivial; set `EXAMPLE_TASK_ID` manually to inspect a specific task.

In [ ]:
if (tasks_aug["n_events"] > 0).any():
    EXAMPLE_TASK_ID = tasks_aug.sort_values("n_events", ascending=False)["task_id"].iloc[0]
else:
    EXAMPLE_TASK_ID = tasks_aug["task_id"].iloc[0]

task_timeline = (
    events_enriched.loc[events_enriched["task_id"] == EXAMPLE_TASK_ID]
    .sort_values(["event_time", "event_order"], na_position="last")
)
print(f"Example task_id: {EXAMPLE_TASK_ID} | n_events={len(task_timeline)}")
show_cols = [c for c in ["event_order", "event_time", "action", "assignee", "status"]
            if c in task_timeline.columns]
display(task_timeline[show_cols].head(50))

## 20. Export Analysis Tables

Write outputs so you can use them in downstream notebooks / scripts / slides.

In [ ]:
OUT_DIR = REPO_ROOT / "notebooks" / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

exports = {
    "tasks_with_event_counts.csv": (tasks_aug, False),
    "differ_events_enriched.csv": (events_enriched, False),
    "proofreader_summary.csv": (proofreader_summary, True),
}
for name, (frame, with_index) in exports.items():
    frame.to_csv(OUT_DIR / name, index=with_index)

print(f"Wrote outputs to: {OUT_DIR}")
for name in exports:
    print(f"  - {name}")

## 21. Interpretation & Talk Frame

**Language of proofreading -> calibrate the people.** The differstack is the
behavioral trace of a proofreading session; Sections 13-16 turn it into measurable
structure:

- **Vocabulary / grammar / dialects (14)** make "some learn to see a neuron and others do not" a *measurement*, not a verdict: proofreaders with converged competence should share grammar and sit close in dialect-distance; novices look higher-entropy and farther apart.
- **Fluency (16)** separates focused work from idle time, pricing the *real* expert minutes a decision costs.
- **The predictive test** (deck slide 10) becomes: does *early* language (first few sessions) predict later agreement / promotion? This notebook produces exactly those early-behavior features per user.

**Analysis ideas to push further:**
- Stratify vocabulary/grammar by `decision` and `flag_reason` to characterize failure modes.
- Correlate dialect-distance with staged-promotion outcomes (the already-deployed "promote on agreement" mechanism).
- Build per-session features (`n_events`, dominant action, grammar entropy, `span_s`, active fraction) as inputs to the pre-registered competence model.
- Re-run Section 3 with a higher `SURVEY_LIMIT`, then loop the pipeline over the top namespaces to quantify the full data anchor (N users x M events x K action types).